In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import RidgeClassifier, LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

print("✅ كل المكتبات تم تحميلها بنجاح!")

✅ كل المكتبات تم تحميلها بنجاح!


In [2]:
df = pd.read_json(r"C:\Users\GigaByte2\Downloads\archive7777\Sarcasm.json", lines=True)

In [3]:
df.head()

,article_link,headline,is_sarcastic
0,https://www.huffingtonpost.com/entry/versace-b...,former versace store clerk sues over secret 'b...,0
1,https://www.huffingtonpost.com/entry/roseanne-...,the 'roseanne' revival catches up to our thorn...,0
2,https://local.theonion.com/mom-starting-to-fea...,mom starting to fear son's web series closest ...,1
3,https://politics.theonion.com/boehner-just-wan...,"boehner just wants wife to listen, not come up...",1
4,https://www.huffingtonpost.com/entry/jk-rowlin...,j.k. rowling wishes snape happy birthday in th...,0


In [4]:
df = df.drop_duplicates(subset=['headline']).reset_index(drop=True)
df = df.drop(columns=['article_link'])

In [5]:
print("Shape:", df.shape)
print("\nتوزيع السخرية:\n", df['is_sarcastic'].value_counts())

Shape: (26603, 2)

توزيع السخرية:
 is_sarcastic
0    14951
1    11652
Name: count, dtype: int64


In [6]:
# ====================== تنظيف + Feature Engineering ======================
def super_clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s!?\'",.:;()–-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_headline'] = df['headline'].apply(super_clean)

df['length'] = df['clean_headline'].str.len()
df['word_count'] = df['clean_headline'].str.split().str.len()
df['excl'] = df['headline'].str.count('!')
df['quest'] = df['headline'].str.count(r'\?')
df['colon'] = df['headline'].str.count(':')
df['quote'] = df['headline'].str.count('"')
df['contains_sarcasm_words'] = df['clean_headline'].str.contains(
    r'report|area man|area woman|study finds|just wants').astype(int)

print("✅ تم التنظيف + Feature Engineering بنجاح")
print(df[['clean_headline', 'length', 'excl']].head())

✅ تم التنظيف + Feature Engineering بنجاح
                                      clean_headline  length  excl
0  former versace store clerk sues over secret 'b...      78     0
1  the 'roseanne' revival catches up to our thorn...      84     0
2  mom starting to fear son's web series closest ...      79     0
3  boehner just wants wife to listen, not come up...      84     0
4  j.k. rowling wishes snape happy birthday in th...      64     0


In [7]:
# ====================== Preprocessor Pipeline ======================
preprocessor = ColumnTransformer([
    ('tfidf', TfidfVectorizer(max_features=40000, ngram_range=(1,4),
                              stop_words=None, min_df=2, sublinear_tf=True),
     'clean_headline'),
    ('num', StandardScaler(), ['length', 'word_count', 'excl', 'quest',
                               'colon', 'quote', 'contains_sarcasm_words'])
])

print("✅ تم إعداد Preprocessor بنجاح")

✅ تم إعداد Preprocessor بنجاح


In [27]:
results = {}
print("✅ تم تهيئة results dictionary بنجاح - جاهز لتسجيل الموديلات")

✅ تم تهيئة results dictionary بنجاح - جاهز لتسجيل الموديلات


In [28]:
# ====================== 1. RidgeClassifier ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

ridge_pipe = Pipeline([('prep', preprocessor), ('clf', RidgeClassifier(alpha=0.8, class_weight='balanced', random_state=42))])
ridge_pipe.fit(X_train, y_train)
pred = ridge_pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='macro')

results["1. RidgeClassifier"] = (acc, f1)

print(f"✅ RidgeClassifier → Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

✅ RidgeClassifier → Accuracy: 0.8585 | F1-Score: 0.8571


In [29]:
# ====================== 2. LinearSVC ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

lsvc_pipe = Pipeline([('prep', preprocessor), ('clf', LinearSVC(class_weight='balanced', random_state=42, max_iter=2000))])
lsvc_pipe.fit(X_train, y_train)
pred = lsvc_pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='macro')

results["2. LinearSVC"] = (acc, f1)

print(f"✅ LinearSVC → Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

✅ LinearSVC → Accuracy: 0.8585 | F1-Score: 0.8569


In [30]:
# ====================== 3. SGDClassifier ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

sgd_pipe = Pipeline([('prep', preprocessor), ('clf', SGDClassifier(loss='hinge', max_iter=1000, tol=1e-3, 
                                                                  class_weight='balanced', random_state=42))])
sgd_pipe.fit(X_train, y_train)
pred = sgd_pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='macro')

results["3. SGDClassifier"] = (acc, f1)

print(f"✅ SGDClassifier → Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

✅ SGDClassifier → Accuracy: 0.8487 | F1-Score: 0.8477


In [31]:
# ====================== 4. LogisticRegression ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

log_pipe = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))])
log_pipe.fit(X_train, y_train)
pred = log_pipe.predict(X_test)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='macro')

results["4. LogisticRegression"] = (acc, f1)

print(f"✅ LogisticRegression → Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

✅ LogisticRegression → Accuracy: 0.8491 | F1-Score: 0.8481


In [32]:
# ====================== 5. MultinomialNB ======================
X_train, X_test, y_train, y_test = train_test_split(
    df, df['is_sarcastic'], test_size=0.2, stratify=df['is_sarcastic'], random_state=42
)

vec = TfidfVectorizer(max_features=40000, ngram_range=(1,4))
X_train_tfidf = vec.fit_transform(X_train['clean_headline'])
X_test_tfidf = vec.transform(X_test['clean_headline'])

nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
pred = nb.predict(X_test_tfidf)

acc = accuracy_score(y_test, pred)
f1 = f1_score(y_test, pred, average='macro')

results["5. MultinomialNB"] = (acc, f1)

print(f"✅ MultinomialNB → Accuracy: {acc:.4f} | F1-Score: {f1:.4f}")

✅ MultinomialNB → Accuracy: 0.8442 | F1-Score: 0.8386


In [33]:
# ====================== الجدول النهائي ======================
data = {"Model": [], "Accuracy": [], "F1-Score": []}

for name, (acc, f1) in results.items():
    clean_name = name.replace("1. ","").replace("2. ","").replace("3. ","").replace("4. ","").replace("5. ","")
    data["Model"].append(clean_name)
    data["Accuracy"].append(acc)
    data["F1-Score"].append(f1)

df_comp = pd.DataFrame(data)
df_comp = df_comp.sort_values(by="Accuracy", ascending=False).reset_index(drop=True)

display(df_comp.style
        .highlight_max(subset=['Accuracy', 'F1-Score'], color='lightgreen')
        .format({"Accuracy": "{:.4f}", "F1-Score": "{:.4f}"})
        .set_caption("📊 Performance Comparison - Sarcasm Detection Models"))

best = df_comp.iloc[0]
print(f"\n🏆 أفضل موديل: {best['Model']}")
print(f"Accuracy  : {best['Accuracy']:.4f}")
print(f"F1-Score  : {best['F1-Score']:.4f}")

,Model,Accuracy,F1-Score
0,RidgeClassifier,0.8585,0.8571
1,LinearSVC,0.8585,0.8569
2,LogisticRegression,0.8491,0.8481
3,SGDClassifier,0.8487,0.8477
4,MultinomialNB,0.8442,0.8386



🏆 أفضل موديل: RidgeClassifier
Accuracy  : 0.8585
F1-Score  : 0.8571
